##### row_number()
- is a **window function** that **assigns a unique, sequential integer** to each row within a **window partition**, **starting from 1**.
- It is commonly used for tasks such as **ranking, identifying top or bottom records**, or simply adding a unique identifier to rows based on a specific order.

**Content:**
- `How to Remove Duplicate Customer Records?`
- `How to filter duplicate records based on multiple primary keys?`
- `How to Use ROW_NUMBER() in PySpark SQL?`

In [0]:
# Import necessary functions for window operations and row numbering
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

##### 1) How to Remove Duplicate Customer Records?

**Scenario:**
- You are working as a Data Engineer in an `marketing company`. Customer data is coming from `multiple source systems` and **due to retries and reprocessing**, `duplicate records` are created.

**Requirement:**

- **Keep only the latest record per customer** based on **load_load**.
- **Remove older duplicate records**.

**Sample Input Data:**

| customer_id |   Name    |  Age |    City     |  load_date   |
| ----------- | --------- | -----| ----------- | ------------ |
| XZ101       | Prasad    |  26  | Hyderabad   | 2025-01-10   |
| XZ101       | Prasad    |  26  | Hyderabad   | 2025-01-15   |
| XZ102       | Swati     |  31  | Noida       | 2025-01-12   |
| XZ103       | Akash     |  24  | Chennai     | 2025-01-11   |
| XZ103       | Akash     |  24  | Chennai     | 2025-01-18   |
| XZ104       | Karishma  |  23  | Bangalore   | 2026-01-22   |
| XZ104       | Karishma  |  23  | Bangalore   | 2026-01-05   |
| XZ104       | Karishma  |  23  | Bangalore   | 2026-01-15   |
| XZ105       | Suhash    |  27  | Amaravati   | 2026-02-05   |

**Expected Output:**

| customer_id |  name     | Age  |   city      |  load_date   |
| ----------- | --------- | ---- | ----------- | -------------|
| XZ101       | Prasad    |  26  | Hyderabad   | 2025-01-15   |
| XZ102       | Swati     |  31  | Noida       | 2025-01-12   |
| XZ103       | Akash     |  24  | Chennai     | 2025-01-18   |
| XZ104       | Karishma  |  23  | Bangalore   | 2026-01-22   |
| XZ105       | Suhash    |  27  | Amaravati   | 2026-02-05   |

In [0]:
# Sample data
data = [
    ("XZ101", "Prasad",	26,	"Hyderabad", "2025-01-10"),
    ("XZ101", "Prasad",	26,	"Hyderabad", "2025-01-15"),
    ("XZ102", "Swati", 31,	"Noida", "2025-01-12"),
    ("XZ103", "Akash", 24, "Chennai", "2025-01-11"),
    ("XZ103", "Akash", 24, "Chennai", "2025-01-18"),
    ("XZ104", "Karishma", 23, "Bangalore", "2026-01-22"),
    ("XZ104", "Karishma", 23, "Bangalore", "2026-01-05"),
    ("XZ104", "Karishma", 23, "Bangalore", "2026-01-15"),
    ("XZ105", "Suhash", 27, "Amaravati", "2026-02-05")
]

columns = ["customer_id", "name", "age", "city", "load_date"]

df_row = spark.createDataFrame(data, columns)
display(df_row)

customer_id,name,age,city,load_date
XZ101,Prasad,26,Hyderabad,2025-01-10
XZ101,Prasad,26,Hyderabad,2025-01-15
XZ102,Swati,31,Noida,2025-01-12
XZ103,Akash,24,Chennai,2025-01-11
XZ103,Akash,24,Chennai,2025-01-18
XZ104,Karishma,23,Bangalore,2026-01-22
XZ104,Karishma,23,Bangalore,2026-01-05
XZ104,Karishma,23,Bangalore,2026-01-15
XZ105,Suhash,27,Amaravati,2026-02-05


In [0]:
# Window specification
window_spec = Window.partitionBy("customer_id") \
                    .orderBy(col("load_date"))

# Remove duplicates
final_df = (
    df_row.withColumn("row_num", row_number().over(window_spec))
        #   .filter(col("row_num") == 1)
        #   .drop("row_num")
)

display(final_df)

customer_id,name,age,city,load_date,row_num
XZ101,Prasad,26,Hyderabad,2025-01-10,1
XZ101,Prasad,26,Hyderabad,2025-01-15,2
XZ102,Swati,31,Noida,2025-01-12,1
XZ103,Akash,24,Chennai,2025-01-11,1
XZ103,Akash,24,Chennai,2025-01-18,2
XZ104,Karishma,23,Bangalore,2026-01-05,1
XZ104,Karishma,23,Bangalore,2026-01-15,2
XZ104,Karishma,23,Bangalore,2026-01-22,3
XZ105,Suhash,27,Amaravati,2026-02-05,1


In [0]:
# Window specification
window_spec = Window.partitionBy("customer_id") \
                    .orderBy(col("load_date").desc())

# Remove duplicates
final_df = (
    df_row.withColumn("row_num", row_number().over(window_spec))
          .filter(col("row_num") == 1)
          .drop("row_num")
)

display(final_df)

customer_id,name,age,city,load_date
XZ101,Prasad,26,Hyderabad,2025-01-15
XZ102,Swati,31,Noida,2025-01-12
XZ103,Akash,24,Chennai,2025-01-18
XZ104,Karishma,23,Bangalore,2026-01-22
XZ105,Suhash,27,Amaravati,2026-02-05


##### 2) Filter duplicate records based on multiple primary keys

In [0]:
%sql
CREATE OR REPLACE TABLE sales_events (
    SALES_ID STRING,
    EVENT_ID STRING,
    TASK_ID STRING,
    PURCHASE_ID STRING,
    CUSTOM_ID STRING,
    AMOUNT DOUBLE,
    EVENT_TIMESTAMP TIMESTAMP
);

INSERT INTO sales_events VALUES
("S1", "E1", "T1", "P1", "C1", 100.0, "2025-01-01 10:00:00"),
("S1", "E1", "T1", "P1", "C1", 100.0, "2025-01-01 10:05:00"),  -- duplicate PK
("S1", "E1", "T1", "P1", "C1", 100.0, "2025-01-01 10:45:45"),  -- duplicate PK    
("S2", "E2", "T2", "P2", "C2", 200.0, "2025-01-02 11:00:00"),
("S3", "E3", "T3", "P3", "C3", 300.0, "2025-01-03 12:00:00"),
("S4", "E4", "T4", "P4", "C4", 150.0, "2025-03-10 15:50:45"),
("S5", "E5", "T5", "P5", "C5", 189.0, "2025-05-15 18:35:10"),
("S5", "E5", "T5", "P5", "C5", 189.0, "2025-05-15 19:55:10"),  -- duplicate PK
("S5", "E5", "T5", "P5", "C5", 189.0, "2025-05-15 20:35:40"),  -- duplicate PK    
("S6", "E6", "T6", "P6", "C6", 333.0, "2025-07-23 15:27:15");

num_affected_rows,num_inserted_rows
10,10


In [0]:
df_task_event = spark.sql("SELECT * FROM sales_events").display()

SALES_ID,EVENT_ID,TASK_ID,PURCHASE_ID,CUSTOM_ID,AMOUNT,EVENT_TIMESTAMP
S1,E1,T1,P1,C1,100.0,2025-01-01T10:00:00.000Z
S1,E1,T1,P1,C1,100.0,2025-01-01T10:05:00.000Z
S1,E1,T1,P1,C1,100.0,2025-01-01T10:45:45.000Z
S2,E2,T2,P2,C2,200.0,2025-01-02T11:00:00.000Z
S3,E3,T3,P3,C3,300.0,2025-01-03T12:00:00.000Z
S4,E4,T4,P4,C4,150.0,2025-03-10T15:50:45.000Z
S5,E5,T5,P5,C5,189.0,2025-05-15T18:35:10.000Z
S5,E5,T5,P5,C5,189.0,2025-05-15T19:55:10.000Z
S5,E5,T5,P5,C5,189.0,2025-05-15T20:35:40.000Z
S6,E6,T6,P6,C6,333.0,2025-07-23T15:27:15.000Z


In [0]:
data = [
    ("S1", "E1", "T1", "P1", "C1", 100.0, "2025-01-01 10:00:00"),
    ("S1", "E1", "T1", "P1", "C1", 100.0, "2025-01-01 10:05:00"),  # duplicate PK
    ("S1", "E1", "T1", "P1", "C1", 100.0, "2025-01-01 10:45:45"),  # duplicate PK    
    ("S2", "E2", "T2", "P2", "C2", 200.0, "2025-01-02 11:00:00"),
    ("S3", "E3", "T3", "P3", "C3", 300.0, "2025-01-03 12:00:00"),
    ("S4", "E4", "T4", "P4", "C4", 150.0, "2025-03-10 15:50:45"),
    ("S5", "E5", "T5", "P5", "C5", 189.0, "2025-05-15 18:35:10"),
    ("S5", "E5", "T5", "P5", "C5", 189.0, "2025-05-15 19:55:10"),  # duplicate PK
    ("S5", "E5", "T5", "P5", "C5", 189.0, "2025-05-15 20:35:40"),  # duplicate PK    
    ("S6", "E6", "T6", "P6", "C6", 333.0, "2025-07-23 15:27:15"),
]

columns = ["SALES_ID", "EVENT_ID", "TASK_ID", "PURCHASE_ID", "CUSTOM_ID", "AMOUNT", "EVENT_TIMESTAMP"]

df_mlt_key = spark.createDataFrame(data, columns) \
    .withColumn("EVENT_TIMESTAMP", F.col("EVENT_TIMESTAMP").cast("timestamp"))

display(df_mlt_key)
df_mlt_key.printSchema()

SALES_ID,EVENT_ID,TASK_ID,PURCHASE_ID,CUSTOM_ID,AMOUNT,EVENT_TIMESTAMP
S1,E1,T1,P1,C1,100.0,2025-01-01T10:00:00.000Z
S1,E1,T1,P1,C1,100.0,2025-01-01T10:05:00.000Z
S1,E1,T1,P1,C1,100.0,2025-01-01T10:45:45.000Z
S2,E2,T2,P2,C2,200.0,2025-01-02T11:00:00.000Z
S3,E3,T3,P3,C3,300.0,2025-01-03T12:00:00.000Z
S4,E4,T4,P4,C4,150.0,2025-03-10T15:50:45.000Z
S5,E5,T5,P5,C5,189.0,2025-05-15T18:35:10.000Z
S5,E5,T5,P5,C5,189.0,2025-05-15T19:55:10.000Z
S5,E5,T5,P5,C5,189.0,2025-05-15T20:35:40.000Z
S6,E6,T6,P6,C6,333.0,2025-07-23T15:27:15.000Z


root
 |-- SALES_ID: string (nullable = true)
 |-- EVENT_ID: string (nullable = true)
 |-- TASK_ID: string (nullable = true)
 |-- PURCHASE_ID: string (nullable = true)
 |-- CUSTOM_ID: string (nullable = true)
 |-- AMOUNT: double (nullable = true)
 |-- EVENT_TIMESTAMP: timestamp (nullable = true)



In [0]:
primary_keys= "SALES_ID|EVENT_ID|TASK_ID|PURCHASE_ID|CUSTOM_ID"
print("input: ", primary_keys)
print("datatype:", type(primary_keys))

lstpKeys = primary_keys.split("|")
print("\noutput: ", lstpKeys)
print("datatype:", type(lstpKeys))

input:  SALES_ID|EVENT_ID|TASK_ID|PURCHASE_ID|CUSTOM_ID
datatype: <class 'str'>

output:  ['SALES_ID', 'EVENT_ID', 'TASK_ID', 'PURCHASE_ID', 'CUSTOM_ID']
datatype: <class 'list'>


In [0]:
from pyspark.sql.functions import row_number

window_spec = Window.partitionBy(*lstpKeys).orderBy(F.col('EVENT_TIMESTAMP').desc())

ranked_data_df = df_mlt_key.withColumn("row_no", F.row_number().over(window_spec))
display(ranked_data_df)

SALES_ID,EVENT_ID,TASK_ID,PURCHASE_ID,CUSTOM_ID,AMOUNT,EVENT_TIMESTAMP,row_no
S1,E1,T1,P1,C1,100.0,2025-01-01T10:45:45.000Z,1
S1,E1,T1,P1,C1,100.0,2025-01-01T10:05:00.000Z,2
S1,E1,T1,P1,C1,100.0,2025-01-01T10:00:00.000Z,3
S2,E2,T2,P2,C2,200.0,2025-01-02T11:00:00.000Z,1
S3,E3,T3,P3,C3,300.0,2025-01-03T12:00:00.000Z,1
S4,E4,T4,P4,C4,150.0,2025-03-10T15:50:45.000Z,1
S5,E5,T5,P5,C5,189.0,2025-05-15T20:35:40.000Z,1
S5,E5,T5,P5,C5,189.0,2025-05-15T19:55:10.000Z,2
S5,E5,T5,P5,C5,189.0,2025-05-15T18:35:10.000Z,3
S6,E6,T6,P6,C6,333.0,2025-07-23T15:27:15.000Z,1


In [0]:
returndf = ranked_data_df.filter(ranked_data_df.row_no > 1)
display(returndf)

SALES_ID,EVENT_ID,TASK_ID,PURCHASE_ID,CUSTOM_ID,AMOUNT,EVENT_TIMESTAMP,row_no
S1,E1,T1,P1,C1,100.0,2025-01-01T10:05:00.000Z,2
S1,E1,T1,P1,C1,100.0,2025-01-01T10:00:00.000Z,3
S5,E5,T5,P5,C5,189.0,2025-05-15T19:55:10.000Z,2
S5,E5,T5,P5,C5,189.0,2025-05-15T18:35:10.000Z,3


In [0]:
# keeping the latest record (row_number() == 1)
returndf = ranked_data_df.filter(ranked_data_df.row_no == 1).drop("row_no")
display(returndf)

SALES_ID,EVENT_ID,TASK_ID,PURCHASE_ID,CUSTOM_ID,AMOUNT,EVENT_TIMESTAMP
S1,E1,T1,P1,C1,100.0,2025-01-01T10:45:45.000Z
S2,E2,T2,P2,C2,200.0,2025-01-02T11:00:00.000Z
S3,E3,T3,P3,C3,300.0,2025-01-03T12:00:00.000Z
S4,E4,T4,P4,C4,150.0,2025-03-10T15:50:45.000Z
S5,E5,T5,P5,C5,189.0,2025-05-15T20:35:40.000Z
S6,E6,T6,P6,C6,333.0,2025-07-23T15:27:15.000Z


##### 3) Use ROW_NUMBER() in PySpark SQL

In [0]:
# Register as a temp SQL table
df_mlt_key.createOrReplaceTempView("source_table")

In [0]:
result_df = spark.sql("""
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY SALES_ID, EVENT_ID, TASK_ID, PURCHASE_ID, CUSTOM_ID
               ORDER BY EVENT_TIMESTAMP DESC
           ) AS row_num
    FROM source_table
""")

result_df.display()

SALES_ID,EVENT_ID,TASK_ID,PURCHASE_ID,CUSTOM_ID,AMOUNT,EVENT_TIMESTAMP,row_num
S1,E1,T1,P1,C1,100.0,2025-01-01T10:45:45.000Z,1
S1,E1,T1,P1,C1,100.0,2025-01-01T10:05:00.000Z,2
S1,E1,T1,P1,C1,100.0,2025-01-01T10:00:00.000Z,3
S2,E2,T2,P2,C2,200.0,2025-01-02T11:00:00.000Z,1
S3,E3,T3,P3,C3,300.0,2025-01-03T12:00:00.000Z,1
S4,E4,T4,P4,C4,150.0,2025-03-10T15:50:45.000Z,1
S5,E5,T5,P5,C5,189.0,2025-05-15T20:35:40.000Z,1
S5,E5,T5,P5,C5,189.0,2025-05-15T19:55:10.000Z,2
S5,E5,T5,P5,C5,189.0,2025-05-15T18:35:10.000Z,3
S6,E6,T6,P6,C6,333.0,2025-07-23T15:27:15.000Z,1


In [0]:
result_df = spark.sql("""
    SELECT * FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY SALES_ID, EVENT_ID, TASK_ID, PURCHASE_ID, CUSTOM_ID
                   ORDER BY EVENT_TIMESTAMP DESC
               ) AS row_num
        FROM source_table
    ) sub
    WHERE row_num = 1
""")

result_df.display()

SALES_ID,EVENT_ID,TASK_ID,PURCHASE_ID,CUSTOM_ID,AMOUNT,EVENT_TIMESTAMP,row_num
S1,E1,T1,P1,C1,100.0,2025-01-01T10:45:45.000Z,1
S2,E2,T2,P2,C2,200.0,2025-01-02T11:00:00.000Z,1
S3,E3,T3,P3,C3,300.0,2025-01-03T12:00:00.000Z,1
S4,E4,T4,P4,C4,150.0,2025-03-10T15:50:45.000Z,1
S5,E5,T5,P5,C5,189.0,2025-05-15T20:35:40.000Z,1
S6,E6,T6,P6,C6,333.0,2025-07-23T15:27:15.000Z,1
